In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

qs = pd.read_csv('/content/drive/MyDrive/qs-world-rankings-2025.csv')

the = pd.read_csv('/content/drive/MyDrive/THE World University Rankings 2016-2026.csv')

In [ ]:
qs.head()
qs.info()
qs.describe()
print(qs.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1503 entries, 0 to 1502
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   2025 Rank                       1503 non-null   object 
 1   2024 Rank                       1482 non-null   object 
 2   Institution Name                1503 non-null   object 
 3   Location                        1503 non-null   object 
 4   Location Full                   1503 non-null   object 
 5   Size                            1503 non-null   object 
 6   Academic Reputation             1503 non-null   float64
 7   Employer Reputation             1503 non-null   float64
 8   Faculty Student                 1503 non-null   float64
 9   Citations per Faculty           1503 non-null   float64
 10  International Faculty           1403 non-null   float64
 11  International Students          1445 non-null   float64
 12  International Research Network  15

In [ ]:
the.info()
the.describe()
print(the.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16713 entries, 0 to 16712
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Rank                     16713 non-null  float64
 1   Name                     16713 non-null  object 
 2   Country                  16713 non-null  object 
 3   Student Population       16713 non-null  object 
 4   Students to Staff Ratio  16713 non-null  float64
 5   International Students   16711 non-null  object 
 6   Female to Male Ratio     15953 non-null  object 
 7   Overall Score            16713 non-null  float64
 8   Teaching                 16713 non-null  float64
 9   Research Environment     16713 non-null  float64
 10  Research Quality         16713 non-null  float64
 11  Industry Impact          16713 non-null  float64
 12  International Outlook    16713 non-null  float64
 13  Year                     16713 non-null  int64  
dtypes: float64(8), int64(1

In [ ]:
qs = qs.drop_duplicates()
the = the.drop_duplicates()

In [ ]:
qs.isnull().sum()
the.isnull().sum()

,0
Rank,0
Name,0
Country,0
Student Population,0
Students to Staff Ratio,0
International Students,2
Female to Male Ratio,760
Overall Score,0
Teaching,0
Research Environment,0


In [ ]:
qs['Institution Name'] = qs['Institution Name'].str.strip().str.lower()
the['Name'] = the['Name'].str.strip().str.lower()
qs['Location'] = qs['Location'].str.strip().str.lower()
the['Country'] = the['Country'].str.strip().str.lower()

In [ ]:
merged = pd.merge(qs, the, left_on=['Institution Name', 'Location'], right_on=['Name', 'Country'], how='outer')

In [ ]:
merged.to_csv('/content/drive/MyDrive/university_cleaned.csv', index=False)

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/university_cleaned.csv')

In [ ]:
# Clean 'Student Population' (from THE rankings in df) and 'International Students_y' (from THE rankings in df)
def clean_population_range(s):
    if pd.isna(s):
        return None
    s = str(s).replace(',', '').strip().lower()
    if s == 'n/a':
        return None
    if '-' in s:
        parts = s.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return None
    elif '+' in s:
        try:
            return float(s.replace('+', ''))
        except ValueError:
            return None
    elif '~' in s:
        try:
            return float(s.replace('~', ''))
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

def clean_percentage_range(s):
    if pd.isna(s):
        return None
    s = str(s).replace('%', '').strip().lower()
    if s == 'n/a':
        return None
    if '-' in s:
        parts = s.split('-')
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except ValueError:
            return None
    try:
        return float(s)
    except ValueError:
        return None

df['Student_Population_Cleaned'] = df['Student Population'].apply(clean_population_range)
df['International_Students_y_Cleaned'] = df['International Students_y'].apply(clean_percentage_range)

df['Global_Rank_Score'] = 100 - df['Rank']   # Using 'Rank' from THE
df['Research_Productivity_Index'] = df['Research Quality'] / df['Students to Staff Ratio']
df['Faculty_Student_Ratio'] = df['Faculty Student']
df['Intl_Student_Percentage'] = df['International_Students_y_Cleaned']

In [ ]:
df[['Student_Population_Cleaned',
    'International_Students_y_Cleaned',
    'Global_Rank_Score',
    'Research_Productivity_Index']].describe()

,Student_Population_Cleaned,International_Students_y_Cleaned,Global_Rank_Score,Research_Productivity_Index
count,1.671300e+04,16703.000000,16713.000000,16713.000000
mean,2.294569e+04,11.232353,-722.676719,3.680942
std,3.268570e+04,12.181542,531.516691,5.249611
min,2.500000e+01,0.000000,-2091.000000,0.017927
25%,9.839000e+03,2.000000,-1107.000000,1.473333
50%,1.741500e+04,7.000000,-660.000000,2.703911
75%,2.886000e+04,16.000000,-280.000000,4.423256
max,1.824383e+06,96.000000,99.000000,248.000000


In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[['Global_Rank_Score',
    'Research_Productivity_Index',
    'Intl_Student_Percentage']] = scaler.fit_transform(
    df[['Global_Rank_Score',
        'Research_Productivity_Index',
        'Intl_Student_Percentage']]
)

In [ ]:
df['Performance_Index'] = (
    0.4 * df['Global_Rank_Score'] +
    0.3 * df['Research_Productivity_Index'] +
    0.3 * df['Intl_Student_Percentage']
)

In [ ]:
import plotly.express as px

fig = px.bar(
    df.sort_values('Performance_Index', ascending=False).head(10),
    x='Institution Name',
    y='Performance_Index',
    title='Top 10 Universities by Performance Index'
)

fig.show()

In [ ]:
import plotly.express as px

df['Performance_Index'] = (
    0.4 * df['Global_Rank_Score'] +
    0.3 * df['Research_Productivity_Index'] +
    0.3 * df['Intl_Student_Percentage']
)

print(df.columns)

fig = px.bar(
    df.sort_values('Performance_Index', ascending=False).head(10),
    x='Institution Name',
    y='Performance_Index',
    title='Top 10 Universities by Performance Index'
)

fig.show()

Index(['2025 Rank', '2024 Rank', 'Institution Name', 'Location',
       'Location Full', 'Size', 'Academic Reputation', 'Employer Reputation',
       'Faculty Student', 'Citations per Faculty', 'International Faculty',
       'International Students_x', 'International Research Network',
       'Employment Outcomes', 'Sustainability', 'QS Overall Score', 'Rank',
       'Name', 'Country', 'Student Population', 'Students to Staff Ratio',
       'International Students_y', 'Female to Male Ratio', 'Overall Score',
       'Teaching', 'Research Environment', 'Research Quality',
       'Industry Impact', 'International Outlook', 'Year',
       'Student_Population_Cleaned', 'International_Students_y_Cleaned',
       'Global_Rank_Score', 'Research_Productivity_Index',
       'Faculty_Student_Ratio', 'Intl_Student_Percentage',
       'Performance_Index'],
      dtype='object')


In [ ]:
df.to_csv('/content/drive/MyDrive/University_final_dataset.csv', index=False)